## Inference mode for AST Model

In [ ]:
import os
import torch
import torchaudio
import torchaudio.transforms as T
from torch import nn
from torchvision import models
from tqdm import tqdm
import numpy as np
import scipy




DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "ast_model_synthetic.pth"
DATA_DIR = "data/trimmed_fan"
SAMPLE_RATE = 16000

# Preprocessing: resample and convert to log-Mel spectrogram
transform = nn.Sequential(
    T.Resample(orig_freq=44100, new_freq=SAMPLE_RATE),
    T.MelSpectrogram(sample_rate=SAMPLE_RATE, n_mels=128),
    T.AmplitudeToDB()
)

class ASTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.vit_b_16(weights="IMAGENET1K_V1")
        #self.backbone.heads = nn.Linear(768, 2)
        #hidden_size = self.backbone.hidden_dim  # Use hidden_dim instead of in_features
        #self.backbone.heads = nn.Linear(hidden_size, 2)
        in_features = self.backbone.heads[0].in_features  # For newer versions
        self.backbone.heads[0] = nn.Linear(in_features, 2)
        #self.backbone.heads = nn.Linear(self.backbone.heads.in_features, 2)

    def forward(self, x):
        return self.backbone(x)

# def load_audio(path):
#     print("failing in beginning of load_audio")
#     waveform, sr = torchaudio.load(path)
#     print("Passing torchaudio load")
#     if waveform.shape[0] > 1:
#         waveform = torch.mean(waveform, dim=0, keepdim=True)
#     if sr != SAMPLE_RATE:
#         waveform = T.Resample(sr, SAMPLE_RATE)(waveform)
#     return waveform

def load_audio(path):
    """Enhanced audio loading with multiple fallbacks"""
    try:
        waveform, sr = torchaudio.load(path)
    except Exception as e:
        try:
            # Try explicitly with soundfile backend
            torchaudio.set_audio_backend("soundfile")
            waveform, sr = torchaudio.load(path)
        except Exception as e2:
            try:
                # Fallback to librosa if available
                import librosa
                y, sr = librosa.load(path, sr=None, mono=True)
                waveform = torch.tensor(y).unsqueeze(0)
            except ImportError:
                # Last resort - read with scipy
                from scipy.io import wavfile
                sr, y = wavfile.read(path)
                if y.ndim > 1:
                    y = np.mean(y, axis=1)
                # Convert to float and normalize if needed
                if y.dtype == np.int16:
                    y = y.astype(np.float32) / 32768.0
                elif y.dtype == np.int32:
                    y = y.astype(np.float32) / 2147483648.0
                waveform = torch.tensor(y).unsqueeze(0)
    
    # Make mono if stereo
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    
    # Resample if needed
    if sr != SAMPLE_RATE:
        resampler = T.Resample(sr, SAMPLE_RATE)
        waveform = resampler(waveform)
    
    return waveform


def preprocess_audio(path):
    waveform = load_audio(path)
    spec = transform(waveform)  # [1, n_mels, time]
    
    # Convert [1, n_mels, time] to [1, 1, n_mels, time] so it's treated as a 2D image
    spec = spec.unsqueeze(1)
    
    # Now interpolate to target size
    spec = torch.nn.functional.interpolate(spec, size=(224, 224))
    
    # Remove the extra dimension
    spec = spec.squeeze(1)
    
    # Normalize
    spec = (spec - spec.mean()) / (spec.std() + 1e-8)
    
    # Expand to 3 channels for ViT
    spec = spec.expand(3, -1, -1)
    
    return spec

# def preprocess_audio(path):
#     waveform = load_audio(path)
#     spec = transform(waveform)  # [1, n_mels, time]
#     spec = torch.nn.functional.interpolate(spec, size=(224, 224))  # Resize for ViT
#     spec = (spec - spec.mean()) / spec.std()  # Normalize
#     spec = spec.expand(3, -1, -1)  # AST expects 3-channel input
#     return spec

def main():
    model = ASTModel().to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    total = 0
    correct = 0

    label_map = {"normal": 0, "abnormal": 1}

    with torch.no_grad():
        for label_str, label_int in label_map.items():
            class_dir = os.path.join(DATA_DIR, label_str)
            for root, _, files in os.walk(class_dir):
                for file in tqdm(files, desc=f"Processing {label_str}"):
                    if not file.endswith(".wav"):
                        continue
                    file_path = os.path.join(root, file)
                    try:
                        input_tensor = preprocess_audio(file_path).unsqueeze(0).to(DEVICE)
                        output = model(input_tensor)
                        pred = torch.argmax(output, dim=1).item()
                        correct += int(pred == label_int)
                        total += 1
                        
                    except Exception as e:
                        print(f"Error processing {file_path}: {e}")

    accuracy = 100.0 * correct / total if total > 0 else 0.0
    print(f"\n✅ Accuracy: {accuracy:.2f}% ({correct}/{total})")
    

if __name__ == "__main__":
    main()


/opt/miniconda3/envs/torchpy311/lib/python3.11/site-packages/torchaudio/functional/functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(
/var/folders/m9/460mzb1j78b5x0bdfn30jtxm0000gn/T/ipykernel_63767/2139290182.py:121: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch

Predicted label: 1
Predicted label: 1
Predicted label: 1


Processing normal:  20%|██        | 6/30 [00:00<00:01, 13.35it/s]

Predicted label: 1
Predicted label: 1
Predicted label: 1


Processing normal:  27%|██▋       | 8/30 [00:00<00:01, 13.82it/s]

...

Predicted label: 1
Predicted label: 1
Predicted label: 1


Processing abnormal:  87%|████████▋ | 26/30 [00:01<00:00, 13.07it/s]

Predicted label: 1
Predicted label: 1
Predicted label: 1


Processing abnormal: 100%|██████████| 30/30 [00:02<00:00, 13.50it/s]

Predicted label: 1
Predicted label: 1
Predicted label: 1

✅ Accuracy: 50.00% (120/240)
